# RevRate - Czyszczenie, Imputacja, Inzynieria cech

## Importy

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)

## Wczytanie danych

In [2]:
df = pd.read_csv('../data/Car_sale_ads.csv')
print('Przed czyszczeniem:', df.shape)

Przed czyszczeniem: (208304, 25)


## Czyszczenie danych

In [3]:
df_cleaned = df.copy()

eur_mask = df_cleaned['Currency'] == 'EUR'
df_cleaned.loc[eur_mask, 'Price'] = (df_cleaned.loc[eur_mask, 'Price'] * 4.30).round().astype(int)

df_cleaned['Price'] = pd.to_numeric(df_cleaned['Price'], errors='coerce')
df_cleaned = df_cleaned[df_cleaned['Price'] > 0].reset_index(drop=True)

Q1 = df_cleaned['Price'].quantile(0.25)
Q3 = df_cleaned['Price'].quantile(0.75)
IQR = Q3 - Q1
df_cleaned = df_cleaned[(df_cleaned['Price'] >= Q1 - 1.5 * IQR) & (df_cleaned['Price'] <= Q3 + 1.5 * IQR)].reset_index(drop=True)

df_cleaned['Production_year'] = pd.to_numeric(df_cleaned['Production_year'], errors='coerce')
df_cleaned = df_cleaned[df_cleaned['Production_year'] >= 1990].reset_index(drop=True)

df_cleaned['Mileage_km'] = pd.to_numeric(df_cleaned['Mileage_km'], errors='coerce')
df_cleaned = df_cleaned[~(df_cleaned['Mileage_km'] < 0)].reset_index(drop=True)
df_cleaned.loc[df_cleaned['Mileage_km'] > 450000, 'Mileage_km'] = 450000
df_cleaned['_age'] = df_cleaned['Production_year'].max() - df_cleaned['Production_year']
used_zero = (df_cleaned['Condition'] == 'Used') & (df_cleaned['Mileage_km'] == 0)
model_age_mean = (
    df_cleaned[df_cleaned['Mileage_km'] > 0]
    .groupby(['Vehicle_model', '_age'])['Mileage_km']
    .mean()
)
idx = pd.MultiIndex.from_frame(df_cleaned.loc[used_zero, ['Vehicle_model', '_age']])
df_cleaned.loc[used_zero, 'Mileage_km'] = idx.map(model_age_mean)
df_cleaned = df_cleaned.drop(columns=['_age'])

df_cleaned['Power_HP'] = pd.to_numeric(df_cleaned['Power_HP'], errors='coerce')
df_cleaned.loc[df_cleaned['Power_HP'] > 800, 'Power_HP'] = 800

df_cleaned['Displacement_cm3'] = pd.to_numeric(df_cleaned['Displacement_cm3'], errors='coerce')

keep_fuel = ['Gasoline', 'Diesel', 'Gasoline + LPG']
df_cleaned = df_cleaned[df_cleaned['Fuel_type'].isin(keep_fuel)].reset_index(drop=True)

df_cleaned['CO2_emissions'] = pd.to_numeric(df_cleaned['CO2_emissions'], errors='coerce')
for ft in keep_fuel:
    mask = df_cleaned['Fuel_type'] == ft
    if mask.sum() > 0:
        cap = df_cleaned.loc[mask, 'CO2_emissions'].quantile(0.99)
        if not pd.isna(cap):
            df_cleaned.loc[mask & (df_cleaned['CO2_emissions'] > cap), 'CO2_emissions'] = cap

df_cleaned['Doors_number'] = pd.to_numeric(df_cleaned['Doors_number'], errors='coerce')
df_cleaned['Doors_number'] = df_cleaned['Doors_number'].clip(2, 5)

df_cleaned = df_cleaned.drop(columns=[
    'Index', 'Currency', 'Offer_location',
    'Vehicle_version', 'Vehicle_generation',
    'First_registration_date', 'First_owner',
    'Features', 'Origin_country', 'CO2_emissions'
])

pd.DataFrame(df_cleaned).to_csv('../data/cleaned_data.csv', index=False)

print('Po czyszczeniu:', df_cleaned.shape)

C:\Users\rukur\AppData\Local\Temp\ipykernel_26140\3858996999.py:28: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_cleaned.loc[used_zero, 'Mileage_km'] = idx.map(model_age_mean)


Po czyszczeniu: (184876, 15)


## Imputacja brakow

In [4]:
def find_similar_cars(pool, core_cols, core_vals, opt_cols, opt_vals):
    cand = pool
    for c, v in zip(core_cols, core_vals):
        cand = cand[cand[c] == v]
    opt_present = [c for c, v in zip(opt_cols, opt_vals) if not pd.isna(v)]
    if opt_present:
        cand_opt = cand
        for c, v in zip(opt_cols, opt_vals):
            if not pd.isna(v):
                cand_opt = cand_opt[cand_opt[c] == v]
        if len(cand_opt) > 0:
            cand = cand_opt
    return cand

def pick_value(cand, col):
    if pd.api.types.is_numeric_dtype(cand[col]):
        return cand[col].median()
    mode_vals = cand[col].mode()
    return mode_vals[0] if not mode_vals.empty else cand.iloc[0][col]

def impute_by_fingerprint(df, target_cols, core_cols, opt_cols):
    n_core = len(core_cols)
    records = []
    for col in target_cols:
        missing = df[df[col].isna()]
        if missing.empty:
            continue
        pool = df[df[col].notna()]
        for key, group in missing.groupby(core_cols + opt_cols, dropna=False):
            core_vals = key[:n_core]
            if any(pd.isna(v) for v in core_vals):
                continue
            opt_vals = key[n_core:]
            cand = find_similar_cars(pool, core_cols, core_vals, opt_cols, opt_vals)
            if len(cand) == 0:
                continue
            opt_present = [c for c, v in zip(opt_cols, opt_vals) if not pd.isna(v)]
            val = pick_value(cand, col)
            for idx in group.index:
                df.loc[idx, col] = val
                records.append({
                    'missing_row': idx,
                    'column': col,
                    'matched_row': cand.index[0],
                    'value': val,
                    'fingerprint_core': str(core_vals),
                    'fingerprint_opt': ','.join(opt_present),
                })
    return records

df_imputed = df_cleaned.copy()

core_cols = ['Vehicle_brand', 'Vehicle_model', 'Production_year', 'Type', 'Fuel_type']
opt_cols = ['Transmission', 'Power_HP', 'Doors_number', 'Displacement_cm3']
target_cols = ['Transmission', 'Doors_number', 'Power_HP', 'Displacement_cm3', 'Drive']

matches = []

for pass_num in range(2):
    pass_start = len(matches)
    matches += impute_by_fingerprint(df_imputed, target_cols, core_cols, opt_cols)
    print(f'Pass {pass_num + 1} fingerprint: {len(matches) - pass_start}')

num_cols = df_imputed.select_dtypes(include=[np.number]).columns
cat_cols = df_imputed.select_dtypes(include=['object']).columns

df_imputed[num_cols] = SimpleImputer(strategy='median').fit_transform(df_imputed[num_cols])
df_imputed[cat_cols] = SimpleImputer(strategy='most_frequent').fit_transform(df_imputed[cat_cols])

pd.DataFrame(matches).to_csv('../data/imputation_matches.csv', index=False)

print('Po imputacji:', df_imputed.shape)

Pass 1 fingerprint: 14642
Pass 2 fingerprint: 0
Po imputacji: (184876, 15)


In [5]:
summary = pd.DataFrame({
    'braki': df_imputed.isnull().sum(),
    '%': (df_imputed.isnull().sum() / len(df_imputed) * 100).round(1),
})
print(summary.sort_values('braki', ascending=False))
assert df_imputed.isnull().sum().sum() == 0

                        braki    %
Price                       0  0.0
Condition                   0  0.0
Vehicle_brand               0  0.0
Vehicle_model               0  0.0
Production_year             0  0.0
Mileage_km                  0  0.0
Power_HP                    0  0.0
Displacement_cm3            0  0.0
Fuel_type                   0  0.0
Drive                       0  0.0
Transmission                0  0.0
Type                        0  0.0
Doors_number                0  0.0
Colour                      0  0.0
Offer_publication_date      0  0.0


## Inzynieria cech

In [6]:
df_features = df_imputed.copy()

pub_dates = pd.to_datetime(df_features['Offer_publication_date'], errors='coerce')
pub_years = pub_dates.dt.year.fillna(pd.Timestamp.now().year).astype(int)

df_features['car_age'] = (pub_years - df_features['Production_year']).clip(lower=0)
df_features.drop(columns=['Production_year', 'Offer_publication_date'], inplace=True)

df_features['mileage_per_year'] = df_features['Mileage_km'] / df_features['car_age'].clip(lower=1)
df_features['power_to_displacement'] = np.where(
    df_features['Displacement_cm3'] > 0,
    df_features['Power_HP'] / df_features['Displacement_cm3'],
    0,
)
df_features['age_x_mileage'] = df_features['car_age'] * df_features['Mileage_km']
print('Ksztalt po ekstrakcji:', df_features.shape)

df_features.to_csv('../data/features_data.csv', index=False)
print('Zapisano features_data.csv')

Ksztalt po ekstrakcji: (184876, 17)
Zapisano features_data.csv
